In [19]:
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, recall_score, precision_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
#from xgboost import XGBClassifier

In [4]:
df = pd.read_csv(r'C:\Users\hugom\projetos\predicao_cardiaco\data\processed\training_data_without_ca.csv')
df

,id,age,trestbps,chol,fbs,thalch,exang,oldpeak,target,sex_Male,cp_atypical angina,cp_non-anginal,cp_typical angina,restecg_normal,restecg_st-t abnormality,slope_flat,slope_upsloping,thal_normal,thal_reversable defect
0,1,63,145.0,233.0,1,150.0,0,2.3,0,1,0,0,1,0,0,0,0,0,0
1,2,67,160.0,286.0,0,108.0,1,1.5,1,1,0,0,0,0,0,1,0,1,0
2,3,67,120.0,229.0,0,129.0,1,2.6,1,1,0,0,0,0,0,1,0,0,1
3,4,37,130.0,250.0,0,187.0,0,3.5,0,1,0,1,0,1,0,0,0,1,0
4,5,41,130.0,204.0,0,172.0,0,1.4,0,0,1,0,0,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
915,916,54,127.0,333.0,1,154.0,0,0.0,1,0,0,0,0,0,1,1,0,1,0
916,917,62,136.0,139.0,0,150.0,0,1.2,0,1,0,0,1,0,1,1,0,1,0
917,918,55,122.0,223.0,1,100.0,0,0.0,1,1,0,0,0,0,1,1,0,0,0
918,919,58,130.0,385.0,1,126.0,1,1.0,0,1,0,0,0,0,0,1,0,0,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        920 non-null    int64  
 1   age                       920 non-null    int64  
 2   trestbps                  920 non-null    float64
 3   chol                      920 non-null    float64
 4   fbs                       920 non-null    int64  
 5   thalch                    920 non-null    float64
 6   exang                     920 non-null    int64  
 7   oldpeak                   920 non-null    float64
 8   target                    920 non-null    int64  
 9   sex_Male                  920 non-null    int64  
 10  cp_atypical angina        920 non-null    int64  
 11  cp_non-anginal            920 non-null    int64  
 12  cp_typical angina         920 non-null    int64  
 13  restecg_normal            920 non-null    int64  
 14  restecg_st

In [6]:
X = df.drop("target", axis=1)
y = df["target"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [7]:
#Utilizando o modelo de logistic Regression
model_lr = LogisticRegression(class_weight="balanced")
model_lr.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced')

In [8]:
y_pred_lr = model_lr.predict(X_test_scaled)

In [9]:
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       0.74      0.81      0.78        75
           1       0.86      0.81      0.83       109

    accuracy                           0.81       184
   macro avg       0.80      0.81      0.81       184
weighted avg       0.81      0.81      0.81       184



In [12]:
#utilizando o modelo Random Forest
model_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_split=2,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=1,
    random_state=42
)
model_rf.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,6
,min_samples_split,2
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [13]:
y_pred_rf = model_rf.predict(X_test)
print(recall_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

0.8899082568807339
              precision    recall  f1-score   support

           0       0.84      0.84      0.84        75
           1       0.89      0.89      0.89       109

    accuracy                           0.87       184
   macro avg       0.86      0.86      0.86       184
weighted avg       0.87      0.87      0.87       184



In [12]:
#Gridsearch para encontrar a profundidade ideal
param_grid = {
    "n_estimators":     [100, 200, 300],
    "max_depth":        [5, 10, 15, None],
    "min_samples_split":[2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features":     ["sqrt", "log2"]
}

model_rf_gs = RandomForestClassifier(
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=model_rf_gs,
    param_grid=param_grid,
    cv=5,
    scoring="recall", #prioridade para ter menos falsos negativos
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)


Fitting 5 folds for each of 216 candidates, totalling 1080 fits


GridSearchCV(cv=5,
             estimator=RandomForestClassifier(class_weight='balanced',
                                              n_jobs=-1, random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': [5, 10, 15, None],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='recall', verbose=2)

In [13]:
#Melhores Parâmetros
print("Melhores Parâmetros:", grid_search.best_params_)
print("Melhor Recall:", grid_search.best_score_)



Melhores Parâmetros: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Melhor Recall: 0.8924999999999998


In [34]:
#identificar overfitting
auc_treino = roc_auc_score(y_train, model_rf.predict_proba(X_train)[:, 1])
auc_teste  = roc_auc_score(y_test,  model_rf.predict_proba(X_test)[:, 1])

print(f"AUC Treino: {auc_treino:.3f}")
print(f"AUC Teste:  {auc_teste:.3f}")
print(f"Diferença:  {auc_treino - auc_teste:.3f}")

AUC Treino: 0.976
AUC Teste:  0.933
Diferença:  0.043


In [31]:
#Encontrar o melhor Threshold
y_proba = model_rf.predict_proba(X_test)[:, 1]
print(f"{'Threshold':<12} {'Recall':<10} {'Precision':<12} {'Falsos Negativos'}")
print("-" * 55)

for t in np.arange(0.25, 0.55, 0.05):
    y_pred = (y_proba >= t).astype(int)
    recall    = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    falsos_negativos = ((y_pred == 0) & (y_test == 1)).sum()
    print(f"{t:<12.2f} {recall:<10.2f} {precision:<12.2f} {falsos_negativos}")

Threshold    Recall     Precision    Falsos Negativos
-------------------------------------------------------
0.25         0.98       0.86         2
0.30         0.97       0.87         3
0.35         0.96       0.87         4
0.40         0.94       0.88         6
0.45         0.92       0.88         9
0.50         0.89       0.89         12
0.55         0.87       0.90         14


In [11]:
#Utilizar o Threshold
THRESHOLD = 0.30

y_proba = model_rf.predict_proba(X_test)[:, 1]
y_pred_final = (y_proba >= THRESHOLD).astype(int)

print(classification_report(y_test, y_pred_final))

              precision    recall  f1-score   support

           0       0.95      0.79      0.86        75
           1       0.87      0.97      0.92       109

    accuracy                           0.90       184
   macro avg       0.91      0.88      0.89       184
weighted avg       0.90      0.90      0.89       184



In [ ]:
#O certo é teinar o modelo e, antes da predição, alterar o Threshold

In [2]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_split=2,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        n_jobs=1,
        random_state=42))
])

In [8]:
pipeline.fit(X_train, y_train)

,steps,"[('scaler', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_estimators,200
,criterion,'gini'
,max_depth,6
,min_samples_split,2


In [9]:
y_pred = pipeline.predict(X_test)
print(recall_score(y_test, y_pred))

0.8899082568807339


In [10]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.84      0.84        75
           1       0.89      0.89      0.89       109

    accuracy                           0.87       184
   macro avg       0.86      0.86      0.86       184
weighted avg       0.87      0.87      0.87       184



In [17]:
joblib.dump(pipeline, r'C:\Users\hugom\projetos\predicao_cardiaco\models\modelo_cardiaco_pipeline.pkl')

['C:\\Users\\hugom\\projetos\\predicao_cardiaco\\models\\modelo_cardiaco_pipeline.pkl']

In [18]:
#dicionário para documentar o threshold
config = {
    "threshold": 0.30,
    "features_esperadas": list(X_train.columns),
    "metrica_otimizada": "recall",
    "recall_no_threshold": 0.97,
    "precision_no_threshold": 0.87
}

In [21]:
with open(r'C:\Users\hugom\projetos\predicao_cardiaco\models\config_modelo.json', 'w') as f:
    json.dump(config, f, indent=2)